# STONE v5 | Comprimento, entropia e detecção

**Pergunta.** A marca separa melhor as respostas geradas com e sem STONE quando há mais posições elegíveis? Como essa diferença varia com a entropia e com o comprimento?

O protocolo usa os 164 problemas do **HumanEval+**, as 3.101 tarefas **call-based** do **APPS** (38 de teste e 3.063 de treino, relatadas separadamente) e as 100 classes do **ClassEval**. Cada tarefa tem duas gerações, com e sem marca, sob a mesma configuração de amostragem. No ClassEval, o modelo recebe o esqueleto e gera a classe inteira de uma vez.

As respostas são guardadas com **todos os IDs gerados e o texto integral**, inclusive eventuais continuações, prosa e tokens especiais. O prompt fica separado. A geração usa EOS ou o limite de contexto do modelo, sem `max_new_tokens` experimental. As métricas usam toda a resposta, excluindo apenas os IDs especiais definidos pelo tokenizer. Os casos que atingem o contexto são incluídos e identificados.

**Colab.** Selecione uma GPU para gerar (T4 ou melhor) e execute em ordem. A configuração padrão percorre todos os problemas e salva cada resposta no Drive. O volume pode exigir várias sessões de GPU, especialmente se houver respostas que não emitam EOS. Para retomar, execute de novo com os mesmos parâmetros. Para inspecionar dados ou resultados sem baixar pesos, escolha `ACAO = 'analisar'`.

### Fontes
- [HumanEval+ v0.1.10](https://github.com/evalplus/humanevalplus_release/releases/tag/v0.1.10)
- [APPS — dados e formato](https://huggingface.co/datasets/codeparrot/apps)
- [ClassEval — dados](https://huggingface.co/datasets/FudanSELab/ClassEval) e [geração holística](https://github.com/FudanSELab/ClassEval)
- [STONE, algoritmos 1 e 2](https://aclanthology.org/2026.findings-eacl.207.pdf)

## 1. Configuração

Para uma verificação rápida, defina `TAREFAS_POR_SESSAO = 2`. Isso pausa a sessão após dois problemas por conjunto, sem alterar a seleção de tarefas do estudo ou o limite de contexto. Volte para `None` para percorrer todos os problemas restantes.

In [ ]:
ACAO = 'gerar_e_analisar'         # ou 'analisar'
USAR_DRIVE = True
TAREFAS_POR_SESSAO = None
MODEL_ID = 'Qwen/Qwen2.5-Coder-3B'
MODEL_REVISION = 'main'          # use um commit fixo para congelar uma nova rodada
SEED = 0
GAMMA, DELTA, HASH_KEY = 0.5, 0.5, 15485863
TOP_K, TEMPERATURA = 50, 1.0
LIMIARES_H = [0.3, 0.6, 0.9, 1.2]
Z_REFERENCIA = 4.0              # fixo para relatar TPR e FPR observadas
B = 1000                       # reamostragens pareadas por tarefa

assert ACAO in {'gerar_e_analisar', 'analisar'}
assert TAREFAS_POR_SESSAO is None or TAREFAS_POR_SESSAO > 0
assert 0 < GAMMA < 1 and DELTA >= 0 and TOP_K >= 1 and TEMPERATURA > 0

In [ ]:
import importlib.metadata as im
import importlib.util, subprocess, sys
versao = '4.57.1'
ja_importado = sys.modules.get('transformers')
if ja_importado is not None and ja_importado.__version__ != versao:
    raise RuntimeError('Reinicie o kernel antes de trocar a versão de Transformers.')
deps = []
try:
    if im.version('transformers') != versao:
        deps.append(f'transformers=={versao}')
except im.PackageNotFoundError:
    deps.append(f'transformers=={versao}')
for mod in ('pandas', 'pyarrow', 'matplotlib', 'scipy', 'tqdm'):
    if importlib.util.find_spec(mod) is None:
        deps.append(mod)
if deps:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *deps])

## 2. Dados e inventário

O APPS é lido linha por linha para não carregar o arquivo completo na RAM. O primeiro download guarda cerca de 1,4 GB de dados brutos do APPS. O critério de inclusão é possuir `fn_name` nos metadados dos testes. Tarefas sem solução de referência disponível continuam na fila. O inventário mostra quantas foram incluídas por divisão. No APPS, o prompt preserva os exemplos que já fazem parte do enunciado. Os arquivos de testes e soluções de referência ficam fora do prompt.

O comprimento de referência serve para formar grupos comuns aos dois braços, antes da geração. Para APPS, usamos a mediana dos comprimentos das soluções disponíveis por problema. Para ClassEval, medimos a classe completa. Para HumanEval+, medimos o trecho a completar. Os grupos são calculados separadamente por conjunto e divisão.

In [ ]:
import os, gc, gzip, json, hashlib, math, re, functools, shutil, time
from pathlib import Path
from urllib.request import urlopen, Request

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import rankdata, spearmanr
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from transformers import LogitsProcessor, LogitsProcessorList

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/stone_pilot/v5')
else:
    ROOT = Path.cwd() / 'stone_v5'
ROOT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / 'dados'
CACHE.mkdir(exist_ok=True)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_gz(path, value):
    tmp = path.with_suffix('.tmp')
    with gzip.open(tmp, 'wt', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False, allow_nan=False, separators=(',', ':'))
    os.replace(tmp, path)

def read_gz(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return json.load(f)

SOURCES = {
    'HumanEval+': 'https://github.com/evalplus/humanevalplus_release/releases/download/v0.1.10/HumanEvalPlus.jsonl.gz',
    'APPS/test': 'https://huggingface.co/datasets/codeparrot/apps/resolve/21e74dd/test.jsonl',
    'APPS/train': 'https://huggingface.co/datasets/codeparrot/apps/resolve/21e74dd/train.jsonl',
    'ClassEval': 'https://huggingface.co/datasets/FudanSELab/ClassEval/resolve/fef204b/data/test-00000-of-00001-5c45fa6e45572491.parquet',
}

def fetch(url, path):
    if path.exists():
        return path
    print('Baixando', path.name, flush=True)
    tmp = path.with_name(path.name + '.part')
    for attempt in range(3):
        try:
            with urlopen(Request(url, headers={'User-Agent':'stone-pilot-v5'}), timeout=90) as r:
                with tmp.open('wb') as f:
                    shutil.copyfileobj(r, f, length=1024*1024)
            os.replace(tmp, path)
            return path
        except (OSError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(2)

cfg = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
revision = cfg._commit_hash or MODEL_REVISION
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision)
CONTEXT = int(cfg.max_position_embeddings)
def ntokens(s):
    return len(tok.encode(s, add_special_tokens=False, verbose=False))

def apps_task(d, split):
    raw = d.get('input_output') or {}
    # Precisamos apenas de fn_name. Entradas dos testes podem ter inteiros de milhares de dígitos.
    io = json.loads(raw, parse_int=str) if isinstance(raw, str) else raw
    fn = (io or {}).get('fn_name')
    if not fn:
        return None
    q = d['question']
    starter = d.get('starter_code') or ''
    reference = d.get('solutions') or []
    if isinstance(reference, str):
        reference = json.loads(reference)
    codes = [s for s in (reference or []) if isinstance(s, str) and s.strip()]
    lens = []
    for i in range(0,len(codes),32):
        batch = tok(codes[i:i+32], add_special_tokens=False, truncation=False, verbose=False)
        lens.extend(map(len,batch.input_ids))
    prompt = f"\nQUESTION:\n{q}\n{starter}\nUse Call-Based Format\n\nANSWER:\n"
    identifier = d['id'] if 'id' in d else d['problem_id']
    return dict(dataset='APPS', split=split, id=str(identifier), prompt=prompt,
                ref_tokens=float(np.median(lens)) if lens else None,
                difficulty=d.get('difficulty') or '', target=fn)

def load_tasks():
    key = digest(dict(sources=SOURCES, model=MODEL_ID, revision=revision, schema=1))[:12]
    index = CACHE / f'inventario_{key}.json.gz'
    if index.exists():
        return read_gz(index)
    tasks = []
    he = fetch(SOURCES['HumanEval+'], CACHE / 'humaneval_v0.1.10.jsonl.gz')
    with gzip.open(he, 'rt', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            tasks.append(dict(dataset='HumanEval+', split='test', id=d['task_id'],
                prompt=d['prompt'], ref_tokens=ntokens(d['canonical_solution']),
                difficulty='', target=d['entry_point']))
    for split in ('test', 'train'):
        file = fetch(SOURCES['APPS/'+split], CACHE / f'apps_21e74dd_{split}.jsonl')
        with file.open(encoding='utf-8') as f:
            n = 0
            for line in tqdm(f, total=5000, desc=f'APPS {split}'):
                n += 1
                task = apps_task(json.loads(line), split)
                if task is not None:
                    tasks.append(task)
        if n != 5000:
            raise ValueError(f'APPS {split}: esperados 5000 registros, encontrados {n}.')
    file = fetch(SOURCES['ClassEval'], CACHE / 'classeval_fef204b.parquet')
    cols = ['task_id', 'skeleton', 'solution_code', 'class_name']
    for d in pd.read_parquet(file, columns=cols).to_dict('records'):
        prompt = ('Implement the complete Python class described by the following skeleton. '
                  'Return the entire class as Python code.\n\n' + d['skeleton'] + '\n\n# Implementation\n')
        tasks.append(dict(dataset='ClassEval', split='test', id=d['task_id'],
            prompt=prompt, ref_tokens=ntokens(d['solution_code']), difficulty='', target=d['class_name']))
    write_gz(index, tasks)
    return tasks

tasks = load_tasks()
groups = {}
for task in tasks:
    task['key'] = digest([task['dataset'], task['split'], task['id']])[:24]
    groups.setdefault((task['dataset'], task['split']), []).append(task)
assert len(groups['HumanEval+', 'test']) == 164 and len(groups['ClassEval', 'test']) == 100
assert all(groups.get(('APPS', s)) for s in ('test', 'train'))
assert len({t['key'] for t in tasks}) == len(tasks), 'IDs duplicados'
inventory = pd.DataFrame(tasks).drop(columns='prompt')
for group, frame in inventory.groupby(['dataset', 'split'], sort=False):
    q1, q2 = frame.ref_tokens.quantile([1/3, 2/3])
    idx = frame.index
    x = frame.ref_tokens
    inventory.loc[idx, 'faixa_ref'] = np.select(
        [x.isna(), x <= q1, x <= q2], ['sem referência', 'curta', 'média'], default='longa')
display(inventory.groupby(['dataset','split'], sort=False).agg(
    tarefas=('key','size'), com_referencia=('ref_tokens','count'),
    mediana_tokens_ref=('ref_tokens','median'), p90_tokens_ref=('ref_tokens',lambda x:x.quantile(.9))))
print('Contexto nativo do modelo:', CONTEXT)

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (4 outputs left out)

APPS train:   0%|          | 0/5000 [00:00<?, ?it/s]

Baixando classeval_fef204b.parquet


tarefas  com_referencia  mediana_tokens_ref  p90_tokens_ref
dataset    split                                                             
HumanEval+ test       164             164               49.50           96.70
APPS       test        38              38              152.75          251.35
           train     3063            3063               68.50          205.00
ClassEval  test       100             100              282.50          477.60

Contexto nativo do modelo: 32768


## 3. Geração integral e registro por posição

O processador segue o modo inspirado no artigo usado na v4: amostra um candidato top-k, aplica a classificação sintática ampla e, quando elegível, adiciona δ aos logits verdes. A lista verde depende do token anterior e é calculada na CPU, como na v4. Esta é a referência experimental do notebook, com as adaptações documentadas.

A entropia é calculada sobre os logits antes da marca e dos filtros de amostragem. A execução sem marca registra H e a elegibilidade hipotética com a mesma regra. A amostragem do candidato usa um gerador aleatório separado para que a instrumentação do controle não consuma os sorteios da geração final.

Cada arquivo comprimido guarda a resposta integral, IDs, H e indicadores alinhados por posição. A execução salva os dois braços separadamente, de modo que uma interrupção entre os braços preserva a resposta já concluída.

In [ ]:
SYN = set(('True False None and as assert async await break class continue def del elif else '
           'except finally for from global if import in is lambda nonlocal not or pass raise return '
           'try while with yield int float complex str bytes bool list tuple set dict NoneType').split())
PUNCT = set('()[]{},:.;@->=+*/%<>!&|^~')

@functools.lru_cache(maxsize=65536)
def syntactic(tid):
    s = tok.decode([tid], skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return not s or s in SYN or all(c in PUNCT for c in s)

@functools.lru_cache(maxsize=128)
def mask_cpu(prev, vocab):
    rng = torch.Generator(device='cpu').manual_seed(HASH_KEY * (prev % vocab))
    green = torch.zeros(vocab, dtype=torch.bool)
    green[torch.randperm(vocab, generator=rng)[:int(vocab*GAMMA)]] = True
    return green

class TraceSTONE(LogitsProcessor):
    def __init__(self, marked, seed, device):
        self.marked = marked
        self.probe = torch.Generator(device=device).manual_seed(seed)
        self.entropies = []
        self.eligible = []

    def __call__(self, input_ids, scores):
        values = scores[0].float()
        logp = torch.log_softmax(values, dim=-1)
        p = logp.exp()
        self.entropies.append(-torch.where(p > 0, p * logp, 0).sum())
        shortlist, ids = torch.topk(values / TEMPERATURA, min(TOP_K, values.numel()))
        sample = torch.multinomial(shortlist.softmax(-1), 1, generator=self.probe)
        eligible = not syntactic(int(ids[sample].item()))
        self.eligible.append(eligible)
        if self.marked and eligible:
            prev = int(input_ids[0, -1].item())
            scores = scores.clone()
            scores[0] += DELTA * mask_cpu(prev, values.numel()).to(scores.device, dtype=scores.dtype)
        return scores

def eos_ids(config):
    eos = config.eos_token_id
    return [eos] if isinstance(eos, int) else list(eos or [])

@torch.inference_mode()
def generate(task, arm):
    enc = tok(task['prompt'], return_tensors='pt', add_special_tokens=False,
              return_token_type_ids=False, truncation=False)
    plen = enc.input_ids.shape[1]
    if not 0 < plen < CONTEXT:
        return dict(status='prompt_sem_espaco_no_contexto', prompt_tokens=plen)
    seed = int(digest([SEED, task['key']])[:8], 16)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    enc = enc.to(model.device)
    proc = TraceSTONE(arm == 'stone', seed ^ 0xA19D, model.device)
    eos = eos_ids(model.generation_config) or [tok.eos_token_id]
    eos = [x for x in eos if x is not None]
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos[0]
    gen = GenerationConfig(max_length=CONTEXT, do_sample=True, temperature=TEMPERATURA,
        top_k=TOP_K, top_p=1.0, eos_token_id=eos, pad_token_id=pad, use_cache=True,
        output_scores=False, forced_eos_token_id=None)
    seq = model.generate(**enc, generation_config=gen,
                         logits_processor=LogitsProcessorList([proc]), use_model_defaults=False)
    ids = seq[0, plen:].tolist()
    assert len(ids) == len(proc.entropies) == len(proc.eligible)
    h = torch.stack(proc.entropies).cpu().tolist() if ids else []
    vocab = model.config.vocab_size
    last = int(enc.input_ids[0,-1].item())
    counted, green = [], []
    special_ids = set(tok.all_special_ids) | set(eos)
    special = [tid in special_ids for tid in ids]
    for tid, is_special in zip(ids, special):
        counted.append(not is_special and not syntactic(tid))
        green.append(bool(mask_cpu(last, vocab)[tid]) if not is_special else False)
        last = tid
    reason = 'eos' if ids and ids[-1] in eos else ('contexto' if plen+len(ids)>=CONTEXT else 'outro')
    return dict(status='ok', prompt_tokens=plen, ids=ids,
        text=tok.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False),
        H=h, eligible=proc.eligible, counted=counted, green=green, special=special, stop=reason)

run_settings = dict(protocol='v5.1', sources=SOURCES, inventory=digest(tasks), model=MODEL_ID,
    revision=revision, context=CONTEXT, gamma=GAMMA, delta=DELTA, hash_key=HASH_KEY,
    temperature=TEMPERATURA, top_k=TOP_K, seed=SEED, transformers=im.version('transformers'))
RUN = ROOT / ('rodada_' + digest(run_settings)[:14])
RUN.mkdir(exist_ok=True)
for dataset, split in groups:
    (RUN / dataset / split).mkdir(parents=True, exist_ok=True)
def checkpoint(task, arm):
    return RUN / task['dataset'] / task['split'] / f"{task['key']}_{arm}.json.gz"
arms = ('sem_marca','stone')
write_gz(RUN / 'config.json.gz', run_settings)
inventory.to_csv(RUN / 'inventario.csv', index=False)
print('Pasta da rodada:', RUN)

Pasta da rodada: /content/drive/MyDrive/stone_pilot/v5/rodada_2a5e0bf9d18083


In [ ]:
pending = [[t for t in ts if any(not checkpoint(t,a).exists() for a in arms)]
           for ts in groups.values()]
if ACAO == 'gerar_e_analisar' and any(pending):
    if not torch.cuda.is_available():
        raise RuntimeError("Ative GPU no Colab para gerar ou selecione ACAO='analisar'.")
    if globals().get('model') is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision,
             dtype=torch.float16, attn_implementation='sdpa').to('cuda').eval()
    # Alterna os conjuntos para que a primeira sessão já produza pares de cada benchmark.
    from itertools import zip_longest
    pending = [ts[:TAREFAS_POR_SESSAO] for ts in pending]
    queue = [t for row in zip_longest(*pending) for t in row if t is not None]
    failures = []
    for t in tqdm(queue, desc='Tarefas pendentes nesta sessão'):
        for arm in arms:
            dest = checkpoint(t, arm)
            if dest.exists():
                continue
            try:
                sample = generate(t, arm)
                write_gz(dest, sample)
            except torch.OutOfMemoryError:
                failures.append([t['dataset'], t['split'], t['id'], arm, 'GPU sem memória'])
                gc.collect()
                torch.cuda.empty_cache()
                print('Memória insuficiente; geração continua pendente:', t['dataset'], t['id'], arm)
    if failures:
        display(pd.DataFrame(failures, columns=['dataset','split','id','braço','erro']))

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Tarefas pendentes nesta sessão:   0%|          | 0/3365 [00:00<?, ?it/s]

## 4. Comprimento × entropia × detecção

- **Comprimento gerado:** contamos os tokens observados em toda a resposta e informamos também o total de IDs, inclusive EOS.
- **Entropia:** calculamos média, mediana e fração com H ≤ 0,9 nas posições elegíveis pelo gerador. A tabela por faixa de H usa o conjunto de tokens finais que o detector conta.
- **Sinal de detecção:** o z-score compara a contagem de tokens verdes à referência γ. Medimos também a diferença pareada em z, AUROC e taxas observadas de verdadeiros e falsos positivos no limiar fixo. Quando não há tokens contáveis, usamos z=0 como na v4 e reportamos esses casos.

As faixas de comprimento **da referência** são definidas antes de observar as gerações. Os cruzamentos com o **comprimento gerado** são descritivos, pois o STONE também pode mudar o comprimento da resposta. APPS treino e teste são relatados separadamente. Priorize o resultado da divisão de teste ao descrever desempenho no benchmark. A comparação principal usa pares completos. A tabela de cobertura inclui pendências e prompts sem espaço.

Os resultados com limiar de H mudam apenas o detector, em saídas já geradas por STONE. A v5 mede detecção e entropia. Testes funcionais e geração híbrida são experimentos separados.

In [ ]:
label = lambda tau: 'stone' if tau is None else f'H>{tau:g}'
rules = [None, *LIMIARES_H]
edges = [-np.inf, .3, .6, .9, 1.2, np.inf]
h_labels = ['H ≤ 0,3','0,3 < H ≤ 0,6','0,6 < H ≤ 0,9','0,9 < H ≤ 1,2','H > 1,2']

def summarize_sample(sample, meta, arm):
    h = np.asarray(sample['H'], dtype=float)
    ordinary = ~np.asarray(sample['special'], dtype=bool)
    eligible = ordinary & np.asarray(sample['eligible'],dtype=bool)
    counted = ordinary & np.asarray(sample['counted'],dtype=bool)
    green = np.asarray(sample['green'],dtype=bool)
    row = dict(**meta, arm=arm, stop=sample['stop'], prompt_tokens=sample['prompt_tokens'],
               ids_gerados=len(h), tokens=int(ordinary.sum()), elegiveis=int(eligible.sum()),
               H_media=float(h[eligible].mean()) if eligible.any() else np.nan,
               H_mediana=float(np.median(h[eligible])) if eligible.any() else np.nan,
               baixa_H=float((h[eligible]<=.9).mean()) if eligible.any() else np.nan)
    for tau in rules:
        mask = counted if tau is None else counted & (h>tau)
        n, k = int(mask.sum()), int(green[mask].sum())
        row['n_'+label(tau)] = n
        row['g_'+label(tau)] = k
        row['z_'+label(tau)] = (k-GAMMA*n)/math.sqrt(GAMMA*(1-GAMMA)*n) if n else 0.
    by_h = []
    for lo, hi, tag in zip(edges, edges[1:], h_labels):
        mask = counted & (h>lo) & (h<=hi)
        by_h.append(dict(**meta, arm=arm, faixa_H=tag,
                        n=int(mask.sum()), verdes=int(green[mask].sum())))
    return row, by_h

meta_by_key = inventory.set_index('key').to_dict('index')
rows, bands, coverage = [], [], []
for t in tqdm(tasks, desc='Lendo resultados'):
    samples = {}
    meta = dict(key=t['key'], **meta_by_key[t['key']])
    for arm in arms:
        fp = checkpoint(t, arm)
        samples[arm] = read_gz(fp) if fp.exists() else {'status':'pendente'}
    coverage.append(dict(**meta, **{a:s['status'] for a,s in samples.items()}))
    if all(s['status']=='ok' for s in samples.values()):
        for arm,s in samples.items():
            row, by_h = summarize_sample(s,meta,arm)
            rows.append(row)
            bands.extend(by_h)
cov = pd.DataFrame(coverage)
cov.to_csv(RUN/'cobertura.csv',index=False)
display(cov.groupby(['dataset','split',*arms], dropna=False).size().rename('tarefas').reset_index())
df = pd.DataFrame(rows)
if df.empty:
    print('Nenhum par completo ainda. O inventário e a cobertura já foram exportados.')
else:
    df.to_csv(RUN/'respostas_metricas.csv',index=False)
    print('Pares completos:', len(df)//2, 'de',len(tasks))
    display(df.groupby(['dataset','split','arm','stop']).size().rename('respostas').reset_index())

In [ ]:
def auc(pos, neg):
    n, m = len(pos), len(neg)
    ranks = rankdata(np.r_[pos, neg])
    return float((ranks[:n].sum()-n*(n+1)/2)/(n*m))

def boot_mean(d, rng):
    if len(d)<2:
        return [np.nan, np.nan]
    parts = [d[rng.integers(0,len(d),size=(min(100,B-i),len(d)))].mean(axis=1)
             for i in range(0,B,100)]
    return np.quantile(np.concatenate(parts), [.025,.975])

if not df.empty:
    pair_cols = ['dataset','split','id','key','faixa_ref','difficulty','ref_tokens']
    wm = df[df.arm=='stone'].drop(columns='arm')
    un = df[df.arm=='sem_marca'].drop(columns='arm')
    pairs = wm.merge(un, on=pair_cols, suffixes=('_wm','_un'), validate='one_to_one')
    pairs['delta_z_stone'] = pairs.z_stone_wm - pairs.z_stone_un
    pairs['comprimento_medio_observado'] = (pairs.tokens_wm+pairs.tokens_un)/2
    pairs.to_csv(RUN/'pares.csv',index=False)
    rng = np.random.default_rng(SEED)
    results = []
    for (dataset,split), subset in pairs.groupby(['dataset','split'],sort=False):
        categories = [('todas',subset), *list(subset.groupby('faixa_ref',sort=False))]
        for faixa, p in categories:
            for tau in rules:
                method = label(tau)
                pos, neg = p['z_'+method+'_wm'].to_numpy(), p['z_'+method+'_un'].to_numpy()
                d = pos-neg
                lo,hi = boot_mean(d,rng)
                results.append(dict(dataset=dataset,split=split,faixa_ref=faixa,detector=method,
                    pares=len(p),tokens_medianos_wm=p.tokens_wm.median(),
                    tokens_medianos_un=p.tokens_un.median(),H_media_wm=p.H_media_wm.mean(),
                    H_media_un=p.H_media_un.mean(),fracao_H_le_09_wm=p.baixa_H_wm.mean(),
                    n_detector_mediana_wm=p['n_'+method+'_wm'].median(),
                    zero_tokens_wm=int((p['n_'+method+'_wm']==0).sum()),
                    zero_tokens_un=int((p['n_'+method+'_un']==0).sum()),
                    z_medio_wm=pos.mean(),z_medio_un=neg.mean(),delta_z=d.mean(),
                    delta_z_ic_low=lo,delta_z_ic_high=hi,
                    AUROC=auc(pos,neg) if len(p)>=2 else np.nan,
                    TPR=(pos>=Z_REFERENCIA).mean(),FPR=(neg>=Z_REFERENCIA).mean()))
    report = pd.DataFrame(results)
    report.to_csv(RUN/'deteccao_por_faixa.csv',index=False)
    display(report[report.detector=='stone'].round(3))
    band_df = pd.DataFrame(bands)
    bh = band_df.groupby(['dataset','split','faixa_ref','faixa_H','arm'],sort=False)[['n','verdes']].sum()
    bh = bh.unstack('arm')
    for arm in arms:
        bh['taxa_verdes',arm] = bh['verdes',arm]/bh['n',arm].replace(0,np.nan)
    bh['excesso_verdes_pp','stone_menos_controle'] = 100*(bh['taxa_verdes','stone']-bh['taxa_verdes','sem_marca'])
    bh.columns = ['_'.join(c) for c in bh.columns]
    bh = bh.reset_index()
    bh.to_csv(RUN/'sinal_por_entropia_e_comprimento.csv',index=False)
    display(bh.round(3))

    # Faixas absolutas de comprimento observado, com o mesmo rótulo para os dois braços.
    observed = pairs[['key','comprimento_medio_observado']].copy()
    observed['faixa_observada'] = pd.cut(observed.comprimento_medio_observado,
        [-np.inf,64,128,256,512,np.inf], labels=['≤64','(64,128]','(128,256]','(256,512]','>512'])
    observed_bands = band_df.merge(observed[['key','faixa_observada']],on='key',validate='many_to_one')
    cross = observed_bands.groupby(['dataset','split','faixa_observada','faixa_H','arm'],
        observed=True)[['n','verdes']].sum().unstack('arm')
    for arm in arms:
        cross['taxa_verdes',arm] = cross['verdes',arm]/cross['n',arm].replace(0,np.nan)
    cross['excesso_verdes_pp','stone_menos_controle'] = 100*(cross['taxa_verdes','stone']-cross['taxa_verdes','sem_marca'])
    cross.columns = ['_'.join(c) for c in cross.columns]
    cross.reset_index().to_csv(RUN/'sinal_por_H_e_tokens_observados.csv',index=False)

    assoc = []
    for (dataset,split), p in pairs.groupby(['dataset','split'],sort=False):
        for x in ['comprimento_medio_observado','ref_tokens','H_media_wm','baixa_H_wm','elegiveis_wm']:
            q = p[[x,'delta_z_stone']].dropna()
            rho = (spearmanr(q.iloc[:,0],q.iloc[:,1]).statistic
                   if len(q)>=3 and q.iloc[:,0].nunique()>1 and q.iloc[:,1].nunique()>1 else np.nan)
            assoc.append(dict(dataset=dataset,split=split,variavel=x,n=len(q),rho=rho))
    assoc_df = pd.DataFrame(assoc)
    assoc_df.to_csv(RUN/'associacoes_descritivas.csv',index=False)
    display(assoc_df.round(3))

In [ ]:
if not df.empty:
    for (dataset,split), p in pairs.groupby(['dataset','split'],sort=False):
        fig, ax = plt.subplots(1,2,figsize=(12,4),layout='constrained')
        for faixa, subset in p.groupby('faixa_ref',sort=False):
            ax[0].scatter(subset.comprimento_medio_observado,subset.delta_z_stone,
                          s=18,alpha=.5,label=faixa)
        ax[0].set(xlabel='Tokens gerados, média do par', ylabel='Δz (STONE − sem marca)',
                  xscale='symlog',title='Comprimento observado e detecção')
        ax[0].axhline(0,color='0.5',lw=1)
        ax[0].legend(title='Faixa da referência')
        heat = bh[(bh.dataset==dataset)&(bh.split==split)].pivot(
            index='faixa_ref',columns='faixa_H',values='excesso_verdes_pp_stone_menos_controle')
        heat = heat.reindex(index=[x for x in ['curta','média','longa','sem referência'] if x in heat.index],
                           columns=h_labels)
        scale = max(1,np.nanmax(np.abs(heat.to_numpy()))) if heat.notna().any().any() else 1
        m = ax[1].imshow(heat.to_numpy(),cmap='RdBu_r',vmin=-scale,vmax=scale,aspect='auto')
        ax[1].set_xticks(range(len(heat.columns)),heat.columns,rotation=28,ha='right')
        ax[1].set_yticks(range(len(heat.index)),heat.index)
        ax[1].set(title='Excesso de verdes vs. controle',ylabel='Comprimento da referência')
        fig.colorbar(m,ax=ax[1],label='Diferença em pontos percentuais')
        fig.suptitle(f'{dataset} / {split} · {len(p)} pares')
        fig.savefig(RUN/f'cruzamento_{dataset}_{split}.png',dpi=150)
        plt.show()
        plt.close(fig)

## Como interpretar e retomar

Compare a AUROC e o Δz por faixa de referência. Use o gráfico por faixa de H para localizar onde a proporção de verdes se separa do controle. Confira as contagens de tokens em cada célula antes de interpretar diferenças em grupos pequenos. Os intervalos de Δz reamostram problemas inteiros com o par preservado. A proporção de verdes em cada célula agrega tokens, dando mais peso às saídas longas.

As saídas integrais podem incluir texto além da solução. Uma associação com comprimento pode refletir esse conteúdo, dificuldade ou diferenças entre tarefas. A coluna de parada identifica quando o contexto encerrou a geração. Os arquivos em `rodada_…` incluem o inventário, a cobertura, o resumo das métricas, os gráficos e as respostas originais comprimidas por problema e braço.

Se a sessão terminar, reabra o notebook com o mesmo modelo, revisão e configuração. Ele percorre os arquivos concluídos e gera os braços pendentes. Se mudar a revisão do modelo, o notebook cria outra pasta de rodada. Em caso de memória insuficiente, use uma GPU com mais VRAM para que as gerações pendentes mantenham a mesma configuração.